In [ ]:
import kagglehub
import os

# Path to the dataset folder
data_dir = "/kaggle/input/datasets/mosapabdelghany/medical-insurance-cost-dataset"

# List all files in the dataset directory
print(os.listdir(data_dir))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(os.path.join(data_dir, "insurance.csv"))
df.head()

In [ ]:
df.shape

In [ ]:
sns.barplot(df["sex"])

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df["sex"] = df["sex"].map({"female":0, "male":1})
df["smoker"] = df["smoker"].map({"no":0, "yes":1})

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = pd.get_dummies(df, columns=["region"], dtype=int)

In [ ]:
df

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
sns.histplot(df["charges"], kde=True)
plt.show()

In [ ]:
df["charges"] = np.log1p(df["charges"])

In [ ]:
df["charges"]

In [ ]:
sns.histplot(df["charges"])
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df["bmi"] = scaler.fit_transform(df[["bmi"]])

In [ ]:
df

In [ ]:
X = df.drop("charges", axis=1)
y = df["charges"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
model = lr.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score

print(f"r2:{r2_score(y_test,y_pred)}")

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBRegressor

In [ ]:
xgbr = XGBRegressor(
    n_estimators = 100,
    learning_rate =0.1,
    max_depth=3,
    random_state = 42
)

In [ ]:
xgbr.fit(X_train,y_train)

In [ ]:
y_pred_xgb = xgbr.predict(X_test)

In [ ]:
print(f"r2_xgbr:{r2_score(y_test,y_pred_xgb)}")

In [ ]:
from xgboost import plot_importance

plot_importance(xgbr)
plt.show()

In [ ]:
import joblib

joblib.dump(xgbr, "medical_insurance_xgboost.pkl")
joblib.dump(X_train.columns.tolist(), "medical_insurance_columns.pkl")
model = joblib.load("medical_insurance_xgboost.pkl")
columns = joblib.load("medical_insurance_columns.pkl")

print(columns)